In [ ]:
import pandas as pd
import altair as alt
import plotly.graph_objects as go

# read code
SPOTIFY = "/content/spotify_data.csv"

# read the data into a data frame
spotify_df = pd.read_csv(SPOTIFY)

# keep random 2500 rows
spotify_df = spotify_df.sample(n=2500, random_state=42)

# convert True/False to 1 and 0
spotify_df["explicit"] = spotify_df["explicit"].astype(int)

spotify_df.head()



,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
31990,31990,4JpToqFup6IoCO8oWFhzWh,St. Vincent,The Greatest Rock Tunes 2022,The Melting Of The Sun,1,257519,0,0.520,0.463,...,-10.429,1,0.0958,0.60900,0.001400,0.4260,0.4690,159.931,4.0,electro
35795,35795,3WwLC6FZsy3DeXOxBIxnep,Forró Boys,"Forró di Katiguria, Vol. 3",Nossa História,38,276240,0,0.717,0.822,...,-4.765,1,0.0334,0.08740,0.000000,0.1160,0.9010,92.998,4.0,forro
890,890,0WSGIwIEqpfe3jGJ5fZsHr,Penny and Sparrow,Finch,Don't Wanna Be Without Ya,58,207762,0,0.651,0.860,...,-7.800,1,0.1380,0.60900,0.006460,0.0735,0.2900,115.000,4.0,acoustic
44850,44850,2gpkmR9oX3Jk6rDI6KUwHj,Glare,Into You,Blank,56,216089,0,0.304,0.682,...,-6.699,0,0.0443,0.00710,0.160000,0.1520,0.0795,158.006,4.0,grunge
52180,52180,3gKpMKkrMUEJvONAhC8zmg,The Dirt Drifters,This Is My Blood,Just Got Tonight,17,221133,0,0.561,0.824,...,-4.584,1,0.0384,0.00265,0.000021,0.3390,0.7310,119.997,4.0,honky-tonk


In [ ]:
top_genres = (
    spotify_df['track_genre']
    .value_counts()
    .nlargest(15)
    .index
)

df_genres = spotify_df[
    spotify_df['track_genre'].isin(top_genres)
]


plot = alt.Chart(df_genres).mark_boxplot().encode(
      y=alt.Y('popularity:Q', title="Popularity"),
      x=alt.X('track_genre:N', title="Genre")
  ).properties(
      width=500,
      height=500,
      title="Popularity Distribution by Genre"
  )
plot.show()
plot.save("boxplot.html")




alt.Chart(...)

In [ ]:


spotify_df["explicit_label"] = spotify_df["explicit"].map({
        0: "Clean",
        1: "Explicit"
    })

counts = (
        spotify_df
        .groupby(["track_genre", "explicit_label"])
        .size()
        .reset_index(name="count")
    )

# Top 10 Clean
top_clean = (
    counts[counts["explicit_label"] == "Clean"]
    .sort_values(by="count", ascending=False)
    .head(10)
)

# Top 10 Explicit
top_explicit = (
    counts[counts["explicit_label"] == "Explicit"]
    .sort_values(by="count", ascending=False)
    .head(10)
)

# Combine them
top_counts = pd.concat([top_clean, top_explicit])

labels = (
        pd.concat([top_counts["track_genre"], top_counts["explicit_label"]])
        .unique()
        .tolist()
    )

label_to_index = {label: i for i, label in enumerate(labels)}

link_colors = top_counts["explicit_label"].map({
    "Clean": "rgba(0, 200, 0, 0.5)",      # green
    "Explicit": "rgba(200, 0, 0, 0.5)"    # red
})

fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=labels
        ),
        link=dict(
            source=top_counts["track_genre"].map(label_to_index),
            target=top_counts["explicit_label"].map(label_to_index),
            value=top_counts["count"],
            color=link_colors
        )
    )])

fig.update_layout(title_text="Spotify Sankey: Genre and Explicit Content")
fig.show()


In [ ]:
# altair scatter plot with top 50 artists within 8 genres filtered in legend (some may be removed)
# danceability vs tempo

top_artists = (
    spotify_df['artists']
    .value_counts()
    .nlargest(50)
    .index
)


df_artists = spotify_df[
    spotify_df['artists'].isin(top_artists)
]


top_genres = (
    df_artists['track_genre']
    .value_counts()
    .nlargest(8)
    .index
)


df_filtered = df_artists[
    df_artists['track_genre'].isin(top_genres)
]


artists_list = sorted(df_filtered['artists'].dropna().unique().tolist())


input_dropdown = alt.binding_select(
    options=[None] + artists_list,
    labels=['All'] + artists_list,
    name='Artist: '
)


selection = alt.selection_point(
    fields=['artists'],
    bind=input_dropdown,
    empty='all'
)


scatter_plot = alt.Chart(df_filtered).mark_circle().encode(
    x='tempo:Q',
    y='danceability:Q',
    tooltip=['artists:N', 'tempo:Q', 'danceability:Q'],
    color=alt.condition(
        selection,
        alt.Color('track_genre:N', title='Genre'),
        alt.value('lightgrey')
    ),
    opacity=alt.condition(selection, alt.value(1), alt.value(0.1))
).add_params(
    selection
)

scatter_plot

alt.Chart(...)

In [ ]:
# interactive histogram comparing energy across genres

# get top 5 genres (based on counts) to avoid overcrowding
top_genres = spotify_df["track_genre"].value_counts().head(5).index.tolist()
genre_df = spotify_df[spotify_df["track_genre"].isin(top_genres)]

# interactive selection
highlight = alt.selection_point(fields=["track_genre"], bind="legend")

hist = alt.Chart(genre_df).mark_bar().encode(
    x=alt.X("energy:Q", bin=alt.Bin(maxbins=25), title="Energy"),
    y=alt.Y("count():Q", title="Number of Songs"),
    color=alt.Color(
        "track_genre:N",
        title="Genre"
    ),
    opacity=alt.condition(highlight, alt.value(0.9), alt.value(0.2)),
    tooltip=[alt.Tooltip("count():Q", title="Count")]
).add_params(
    highlight
).properties(
    title="Distribution of Song Energy Across Top Genres",
    width=800,
    height=400
)

hist.save("energy_histogram.html")
hist

alt.Chart(...)